# Nationwide multi-day AQI early-warning system

This notebook trains **cost-sensitive, multi-horizon classifiers** to forecast whether an Indian urban area will exceed AQI **200** (health alert) or **300** (severe smog) in **1, 2, or 3 days**.

It uses the nationwide AQI?weather dataset in this project (291 areas, 2022-04-01 to 2025-04-30). Every dynamic predictor is lagged, so the model never uses the AQI, weather, or pollutant label from the day it is forecasting.

> **Research scope and operational caveat:** NASA POWER is historical reanalysis, not a weather forecast feed. This is therefore a valid historical back-test of an early-warning *modelling framework*, but a live deployment must replace weather lags/reanalysis with weather forecasts available at issue time. GRAP is statutory for Delhi-NCR; nationwide alerts below are health-risk indicators, not legal GRAP declarations.


## 1. Install dependencies (run once)

Uncomment and run the next cell only if the notebook kernel does not already have these packages. Restart the kernel after installation if requested.


In [ ]:
# %pip install -U pandas numpy scikit-learn xgboost matplotlib seaborn joblib


## 2. Configuration

The chronological partitions ensure that model selection, threshold selection, and the final test evaluation use separate time periods. Lower `N_ESTIMATORS` for a faster exploratory run.


In [ ]:
from pathlib import Path
import json
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    average_precision_score, balanced_accuracy_score, confusion_matrix,
    precision_recall_curve, precision_score, recall_score, roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)

# Locate the project whether Jupyter was started in the project root or notebooks/.
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "data" / "processed").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "aqi_prediction_full.csv"
REPORTS_DIR = PROJECT_ROOT / "reports" / "early_warning"
MODELS_DIR = PROJECT_ROOT / "models" / "early_warning"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

HORIZONS = [1, 2, 3]
THRESHOLDS = [200, 300]
TRAIN_END = pd.Timestamp("2024-04-30")
VALID_END = pd.Timestamp("2024-10-31")
TEST_END = pd.Timestamp("2025-04-27")  # leaves room for t+3 labels
MIN_RECALL = 0.80
COST_MULTIPLIER = 2.0       # prioritises missed hazardous events over false alarms
N_ESTIMATORS = 300          # change to 100 for a quicker exploratory run
RANDOM_STATE = 42

assert DATA_PATH.exists(), f"Dataset not found: {DATA_PATH}"
print(f"Project root: {PROJECT_ROOT}")
print(f"Input dataset: {DATA_PATH}")


## 3. Load and validate the source data

`aqi_prediction_full.csv` already contains weather, static geography, and calendar fields. This notebook deliberately does **not** use `air_quality_status`, `prominent_pollutants`, same-day weather, or same-day AQI as features because they leak information about the target day.


In [ ]:
raw = pd.read_csv(DATA_PATH, parse_dates=["date"])
required = {"date", "state", "area", "aqi_value", "latitude", "longitude"}
missing = required - set(raw.columns)
assert not missing, f"Missing required columns: {missing}"

raw["location_key"] = raw["state"].astype(str).str.strip() + "||" + raw["area"].astype(str).str.strip()
raw = raw.sort_values(["location_key", "date"]).drop_duplicates(["location_key", "date"]).reset_index(drop=True)

print(f"Rows: {len(raw):,}")
print(f"Locations: {raw.location_key.nunique():,}; states/UTs: {raw.state.nunique():,}")
print(f"Dates: {raw.date.min():%Y-%m-%d} to {raw.date.max():%Y-%m-%d}")
print("Observed event prevalence:")
display(pd.DataFrame({
    "AQI > 200": [(raw.aqi_value > 200).sum(), (raw.aqi_value > 200).mean()],
    "AQI > 300": [(raw.aqi_value > 300).sum(), (raw.aqi_value > 300).mean()],
}, index=["days", "share"]).T)


## 4. Build leakage-safe forecasting features

For an issue date *t*, lag features contain only observations from *t?1* or earlier. A target is retained only when the next observed record is exactly `horizon` calendar days later; this prevents a missing date from being silently treated as the next day.


In [ ]:
WEATHER_COLS = [
    "temperature_2m_mean", "temperature_2m_max", "temperature_2m_min",
    "relative_humidity_2m_mean", "precipitation_sum",
    "wind_speed_2m_mean", "wind_speed_2m_max",
    "surface_pressure_mean", "shortwave_radiation_sum",
]
WEATHER_COLS = [c for c in WEATHER_COLS if c in raw.columns]

def add_forecast_features(frame: pd.DataFrame, horizons: list[int]) -> tuple[pd.DataFrame, list[str]]:
    out = frame.copy().sort_values(["location_key", "date"]).reset_index(drop=True)
    group = out.groupby("location_key", sort=False)

    # AQI persistence, trend and recent extremes, all ending at t-1.
    for lag in (1, 2, 3, 7, 14):
        out[f"aqi_lag_{lag}"] = group["aqi_value"].shift(lag)
        lag_date = group["date"].shift(lag)
        out.loc[(out["date"] - lag_date).dt.days != lag, f"aqi_lag_{lag}"] = np.nan

    for window in (3, 7, 14):
        prior = group["aqi_value"].shift(1)
        out[f"aqi_mean_{window}d"] = prior.groupby(out["location_key"]).transform(
            lambda s: s.rolling(window, min_periods=window).mean()
        )
        out[f"aqi_max_{window}d"] = prior.groupby(out["location_key"]).transform(
            lambda s: s.rolling(window, min_periods=window).max()
        )
        history_date = group["date"].shift(window)
        invalid = (out["date"] - history_date).dt.days != window
        out.loc[invalid, [f"aqi_mean_{window}d", f"aqi_max_{window}d"]] = np.nan

    # Weather lags: historical information only. Live deployment should provide forecast weather here.
    for column in WEATHER_COLS:
        for lag in (1, 3, 7):
            name = f"{column}_lag_{lag}"
            out[name] = group[column].shift(lag)
            lag_date = group["date"].shift(lag)
            out.loc[(out["date"] - lag_date).dt.days != lag, name] = np.nan

    # The future calendar is known when issuing a forecast.
    for horizon in horizons:
        forecast_date = out["date"] + pd.Timedelta(days=horizon)
        out[f"forecast_month_h{horizon}"] = forecast_date.dt.month
        out[f"forecast_doy_sin_h{horizon}"] = np.sin(2 * np.pi * forecast_date.dt.dayofyear / 365.25)
        out[f"forecast_doy_cos_h{horizon}"] = np.cos(2 * np.pi * forecast_date.dt.dayofyear / 365.25)
        out[f"forecast_weekend_h{horizon}"] = forecast_date.dt.dayofweek.isin([5, 6]).astype(int)

        target = group["aqi_value"].shift(-horizon)
        target_date = group["date"].shift(-horizon)
        target = target.where((target_date - out["date"]).dt.days.eq(horizon))
        out[f"target_aqi_h{horizon}"] = target
        for threshold in THRESHOLDS:
            out[f"event_gt_{threshold}_h{horizon}"] = (target > threshold).astype("float")
            out.loc[target.isna(), f"event_gt_{threshold}_h{horizon}"] = np.nan

    # Static spatial fields and categorical location identifiers are safe at any horizon.
    numeric = [c for c in out.columns if c.startswith(("aqi_lag_", "aqi_mean_", "aqi_max_"))]
    numeric += [c for c in out.columns if any(c.startswith(f"{w}_lag_") for w in WEATHER_COLS)]
    numeric += ["latitude", "longitude"]
    return out, numeric

model_frame, NUMERIC_BASE = add_forecast_features(raw, HORIZONS)
print(f"Feature-engineered rows: {len(model_frame):,}")
print(f"Numeric base features: {len(NUMERIC_BASE)}")


## 5. Cost-sensitive model and threshold-selection functions

`scale_pos_weight` makes rare alert days more costly to miss. The classification threshold is selected on the validation period?not the test period?to meet the configured minimum recall while keeping precision as high as possible.


In [ ]:
CATEGORICAL = ["state", "location_key"]

def make_pipeline(scale_pos_weight: float) -> Pipeline:
    preprocessor = ColumnTransformer(
        transformers=[
            ("numeric", Pipeline([(
                "imputer", SimpleImputer(strategy="median", add_indicator=True)
            )]), NUMERIC_BASE),
            ("categorical", Pipeline([(
                "imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore")),
            ]), CATEGORICAL),
        ]
    )
    classifier = XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        n_estimators=N_ESTIMATORS,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.85,
        colsample_bytree=0.8,
        min_child_weight=3,
        reg_alpha=0.1,
        reg_lambda=1.0,
        scale_pos_weight=scale_pos_weight,
        tree_method="hist",
        n_jobs=-1,
        random_state=RANDOM_STATE,
    )
    return Pipeline([("preprocess", preprocessor), ("model", classifier)])

def choose_alert_threshold(y_true: pd.Series, probabilities: np.ndarray, min_recall: float = MIN_RECALL) -> float:
    candidates = np.round(np.arange(0.05, 0.96, 0.01), 2)
    rows = []
    for threshold in candidates:
        predicted = probabilities >= threshold
        rows.append((threshold, recall_score(y_true, predicted, zero_division=0),
                     precision_score(y_true, predicted, zero_division=0)))
    table = pd.DataFrame(rows, columns=["threshold", "recall", "precision"])
    eligible = table[table.recall >= min_recall]
    if not eligible.empty:
        return float(eligible.sort_values(["precision", "threshold"], ascending=[False, False]).iloc[0].threshold)
    return float(table.sort_values(["recall", "precision"], ascending=False).iloc[0].threshold)

def score_alerts(y_true: pd.Series, probabilities: np.ndarray, threshold: float) -> tuple[dict, np.ndarray]:
    predicted = probabilities >= threshold
    tn, fp, fn, tp = confusion_matrix(y_true, predicted, labels=[0, 1]).ravel()
    metrics = {
        "roc_auc": roc_auc_score(y_true, probabilities) if y_true.nunique() == 2 else np.nan,
        "average_precision": average_precision_score(y_true, probabilities) if y_true.nunique() == 2 else np.nan,
        "recall": recall_score(y_true, predicted, zero_division=0),
        "precision": precision_score(y_true, predicted, zero_division=0),
        "balanced_accuracy": balanced_accuracy_score(y_true, predicted),
        "true_positives": int(tp), "false_negatives": int(fn),
        "false_positives": int(fp), "true_negatives": int(tn),
    }
    return metrics, predicted


## 6. Train and evaluate all six early-warning tasks

- **Train:** through 2024-04-30
- **Validation / alert-threshold selection:** 2024-05-01 to 2024-10-31
- **Unseen final test:** 2024-11-01 to 2025-04-27

Each task has its own horizon and severity threshold. Models are saved to `models/early_warning/`; test predictions and metrics go to `reports/early_warning/`.


In [ ]:
FEATURES = NUMERIC_BASE + CATEGORICAL
results = []
test_predictions = []
models = {}

for horizon in HORIZONS:
    target_column = f"target_aqi_h{horizon}"
    task_frame = model_frame[model_frame[target_column].notna()].copy()
    train = task_frame[task_frame.date <= TRAIN_END]
    valid = task_frame[(task_frame.date > TRAIN_END) & (task_frame.date <= VALID_END)]
    test = task_frame[(task_frame.date > VALID_END) & (task_frame.date <= TEST_END)]

    print(f"\nHorizon t+{horizon}: train={len(train):,}, validation={len(valid):,}, test={len(test):,}")
    for threshold in THRESHOLDS:
        label = f"event_gt_{threshold}_h{horizon}"
        y_train = train[label].astype(int)
        y_valid = valid[label].astype(int)
        y_test = test[label].astype(int)
        positives = int(y_train.sum())
        negatives = int((1 - y_train).sum())
        if positives == 0 or y_valid.nunique() < 2 or y_test.nunique() < 2:
            print(f"  AQI>{threshold}: skipped (insufficient event variation)")
            continue

        class_weight = (negatives / positives) * COST_MULTIPLIER
        pipeline = make_pipeline(class_weight)
        pipeline.fit(train[FEATURES], y_train)

        valid_prob = pipeline.predict_proba(valid[FEATURES])[:, 1]
        operating_threshold = choose_alert_threshold(y_valid, valid_prob)
        test_prob = pipeline.predict_proba(test[FEATURES])[:, 1]
        metrics, predicted = score_alerts(y_test, test_prob, operating_threshold)

        task_name = f"aqi_gt_{threshold}_t_plus_{horizon}"
        joblib.dump(pipeline, MODELS_DIR / f"{task_name}.joblib")
        models[task_name] = pipeline

        row = {
            "task": task_name, "horizon_days": horizon, "aqi_threshold": threshold,
            "train_rows": len(train), "validation_rows": len(valid), "test_rows": len(test),
            "train_positive_events": positives, "train_event_rate": positives / len(train),
            "scale_pos_weight": class_weight, "alert_probability_threshold": operating_threshold,
            **metrics,
        }
        results.append(row)

        pred = test[["date", "state", "area", "location_key", target_column]].copy()
        pred["forecast_date"] = pred["date"] + pd.Timedelta(days=horizon)
        pred["task"] = task_name
        pred["aqi_threshold"] = threshold
        pred["event_observed"] = y_test.to_numpy()
        pred["event_probability"] = test_prob
        pred["alert_probability_threshold"] = operating_threshold
        pred["alert_issued"] = predicted.astype(int)
        test_predictions.append(pred)

        print(
            f"  AQI>{threshold}: recall={metrics['recall']:.3f}, "
            f"precision={metrics['precision']:.3f}, AP={metrics['average_precision']:.3f}, "
            f"threshold={operating_threshold:.2f}"
        )

metrics_df = pd.DataFrame(results).sort_values(["aqi_threshold", "horizon_days"])
predictions_df = pd.concat(test_predictions, ignore_index=True)
metrics_df.to_csv(REPORTS_DIR / "multi_horizon_metrics.csv", index=False)
predictions_df.to_csv(REPORTS_DIR / "multi_horizon_test_predictions.csv", index=False)
with (REPORTS_DIR / "run_configuration.json").open("w", encoding="utf-8") as handle:
    json.dump({
        "input": str(DATA_PATH.relative_to(PROJECT_ROOT)),
        "horizons": HORIZONS, "event_thresholds": THRESHOLDS,
        "train_end": str(TRAIN_END.date()), "validation_end": str(VALID_END.date()),
        "test_end": str(TEST_END.date()), "minimum_validation_recall": MIN_RECALL,
        "cost_multiplier": COST_MULTIPLIER, "n_estimators": N_ESTIMATORS,
    }, handle, indent=2)

display(metrics_df)


## 7. Inspect final-test performance

Recall is the primary safety metric: it is the share of real high-AQI days for which an alert was issued. Precision shows the fraction of alerts that corresponded to a real event. Both must be reported, along with false alarms and missed events.


In [ ]:
plot_data = metrics_df.melt(
    id_vars=["task", "horizon_days", "aqi_threshold"],
    value_vars=["recall", "precision", "average_precision", "balanced_accuracy"],
    var_name="metric", value_name="score",
)
plt.figure(figsize=(12, 6))
sns.barplot(data=plot_data, x="task", y="score", hue="metric")
plt.ylim(0, 1.05)
plt.xlabel("Forecast task")
plt.ylabel("Final-test score")
plt.xticks(rotation=20, ha="right")
plt.title("Multi-horizon early-warning performance on the unseen test period")
plt.tight_layout()
plt.savefig(REPORTS_DIR / "multi_horizon_performance.png", dpi=160)
plt.show()

display(metrics_df[[
    "task", "recall", "precision", "average_precision", "roc_auc",
    "false_negatives", "false_positives", "alert_probability_threshold",
]].round(3))


## 8. Example alert feed and health-risk interpretation

The table shows the highest-risk test-period forecasts. A future operational system would generate this table from the most recent issue date, then deliver it to public-health and local-government users. AQI >200 is treated as a health alert and AQI >300 as severe-smog escalation.


In [ ]:
def risk_message(row: pd.Series) -> str:
    if row["aqi_threshold"] == 300:
        return "Severe-smog early warning (forecast AQI > 300)"
    return "Health-risk early warning (forecast AQI > 200)"

alert_feed = predictions_df[predictions_df.alert_issued.eq(1)].copy()
alert_feed["risk_message"] = alert_feed.apply(risk_message, axis=1)
alert_feed = alert_feed.sort_values(["forecast_date", "event_probability"], ascending=[False, False])
alert_feed.to_csv(REPORTS_DIR / "issued_alerts_test_period.csv", index=False)
display(alert_feed[[
    "date", "forecast_date", "state", "area", "task", "event_probability",
    "event_observed", "risk_message"
]].head(25))


## 9. Reproducibility and paper-reporting checklist

Use the exported files for tables and figures. Before making research claims, report: data provenance; missing-weather handling; chronological partitions; event prevalence by horizon; precision, recall, PR-AUC, false negatives and false positives; area/state subgroup performance; and confidence intervals (for example, block bootstrap by date).

Do **not** describe this system as an operational 72-hour forecast until its features are supplied from data available in real time?particularly numerical weather predictions rather than NASA POWER reanalysis.


In [ ]:
print("Saved outputs")
for path in sorted(REPORTS_DIR.iterdir()):
    print(" -", path.relative_to(PROJECT_ROOT))
print("Saved model files:", len(list(MODELS_DIR.glob("*.joblib"))))
